# 05. Признаки компонент и правила отбора

| вход | выход |
|---|---|
| `closed_mask` из этапа 04 и маска штрихов из этапа 02 | маска детали `part_mask` |

Тип ребра плохо предсказывает, нужен ли объект: зигзаг резьбы попадает в изоляты только потому, что
не связан с деталью, а рамка вида становится набором шпор только потому, что где-то разомкнута.
Поэтому решение принимается по связной компоненте и ее измеримым свойствам.

Правила такие. Компонента умирает, если ее габарит не больше характерного куска текстовой области:
детектор находит не весь текст, но задает масштаб шрифта на листе, и пропущенные буквы совпадают с
ним по размеру. Дальше компонента живет, если у нее есть внутренняя площадь, либо она ровная по толщине и
при этом извилистая; текст внутри контура убивает компоненту в любом случае. Повторяющиеся
незамкнутые компоненты одного размера это шаблон, то есть размерные стрелки. Наконечники, приклеенные
к детали, компонентой не отделяются и срезаются как хвосты, короткие по сравнению со своей
компонентой.

Итоговая маска рисуется по оригинальной бинаризации, поэтому искусственные перемычки замыкания в нее
не попадают.

In [1]:
from pathlib import Path

import numpy as np
from artifact_cache import ArtifactCache
from ipywidgets import Dropdown, FloatSlider, IntSlider, interactive
from stage_metrics import build_component_feature_table, measure_mask_metrics
from visualization import (
    build_view_window,
    draw_component_selection,
    draw_trimmed_twigs,
    show_mask,
    window_sliders,
)

from draftslice.common_types import Mask, Mat
from draftslice.features import (
    ComponentFeatures,
    build_component_features,
    expand_selection_to_paths,
    find_enclosed_components,
    find_host_contours,
    find_repeated_components,
    find_text_sized_components,
    group_similar_components,
    measure_text_component_scale,
    select_components,
    trim_short_twigs,
)
from draftslice.image_preparation import load_bgr_image, scale_image
from draftslice.stroke_graph import StrokeGraph, build_stroke_graph
from draftslice.stroke_painting import paint_selected_paths

DATASET_FOLDER = Path("/mnt/data/datasets/Сравнение чертежей/Эскизы датасет")

## Вход из этапов 02 и 04

In [2]:
cache = ArtifactCache()
source_files: list[Path] = [
    path for path in sorted(DATASET_FOLDER.glob("*.jpg")) if cache.has_stage(path, "closed_mask")
]
print(f"файлов с замкнутой маской: {len(source_files)}")


def run_source(file_index: int) -> Path:
    """Выбор файла среди посчитанных на этапе 04."""
    source_path = source_files[file_index]
    stored = cache.load_latest(source_path, "closed_mask")
    print(f"артефакт: {stored.path.name}")
    print(f"параметры этапа 04: {stored.parameters}")
    return source_path


ui_source = interactive(
    run_source,
    file_index=Dropdown(
        options=[(f"[{index}] {path.stem}", index) for index, path in enumerate(source_files)],
        value=0,
        description="файл",
    ),
)
ui_source

файлов с замкнутой маской: 3


interactive(children=(Dropdown(description='файл', options=(('[0] 1', 0), ('[1] 13', 1), ('[2] 9', 2)), value=…

In [3]:
source_path: Path = ui_source.result
stored_closed = cache.load_latest(source_path, "closed_mask")
closed_mask: Mask = stored_closed.arrays["closed_mask"]
text_region_mask: Mask = stored_closed.arrays["text_region_mask"]
original_strokes_mask: Mask = cache.load_latest(source_path, "strokes").arrays["strokes_mask"]

source_image: Mat = load_bgr_image(source_path)
scaled_image: Mat = scale_image(source_image, closed_mask.shape[1] / source_image.shape[1])

graph: StrokeGraph = build_stroke_graph(closed_mask, text_region_mask)
all_paths: Mask = np.ones(graph.path_count, dtype=bool)
features: ComponentFeatures = build_component_features(graph, all_paths, text_region_mask)
print(f"ребер {graph.path_count} | компонент {len(features)} | висячих ребер {int(features.hanging_path.sum())}")

ребер 239 | компонент 86 | висячих ребер 144


## Паспорт компонент

Смотрим на числа до того, как ставим пороги: у наконечника разброс толщины в разы больше единицы, у
зигзага резьбы высокая извилистость при ровной толщине, у детали большая внутренняя площадь и малая
доля хвостов.

In [4]:
feature_table = build_component_feature_table(features)
print(f"компонент {len(feature_table)}")
print(feature_table.head(20).to_string(index=False))

компонент 86
   id  ребер  длина  габарит  извил  толщина  разброс t  внутри  текст  хвостов  доля хвостов  длиннейший хвост
 2420     19 3296.0    742.0   4.44      5.0       1.24  223232   0.00        2          0.02              37.0
 4266      8 2653.0    736.0   3.60      5.0       1.06  225364   0.00        1          0.00               4.0
15714     11 2306.0    556.0   4.15      5.0       1.00  100202   0.00        1          0.01              33.0
 5905     17 2013.0    395.0   5.10      5.3       1.40   60012   0.00        1          0.01              17.0
 1481      1 1546.0    690.0   2.24      5.0       5.00   59479   0.00        0          0.00               0.0
  505      1  814.0    689.0   1.18      5.0       1.00       0   0.00        1          1.00             814.0
15253      4  633.0    466.0   1.36      5.0       1.00       0   0.00        2          0.99             627.0
12281     10  567.0    144.0   3.95      5.3       1.50    7316   0.00        2          0.

## Правило отбора компонент

In [5]:
def run_component_rules(
    center_x: float,
    center_y: float,
    zoom: float,
    minimum_inside_area: int,
    maximum_thickness_spread: float,
    minimum_tortuosity: float,
    repeat_tolerance: float,
    minimum_repeat_count: int,
    text_size_quantile: float,
    text_size_tolerance: float,
    host_area_share: float,
) -> Mask:
    """Слайдеры правил отбора компонент."""
    kept_components = select_components(
        features,
        minimum_inside_area=minimum_inside_area,
        maximum_thickness_spread=maximum_thickness_spread,
        minimum_tortuosity=minimum_tortuosity,
    )
    groups = group_similar_components(features, repeat_tolerance)
    repeated = find_repeated_components(features, groups, minimum_repeat_count)
    text_scale = measure_text_component_scale(text_region_mask, text_size_quantile)
    text_sized = find_text_sized_components(features, text_scale, text_size_tolerance)
    hosts = find_host_contours(features, host_area_share)
    enclosed = find_enclosed_components(graph, features, hosts)
    survivors = kept_components & ~repeated & ~(text_sized & ~enclosed)
    kept_paths = expand_selection_to_paths(features, survivors)

    print(
        f"компонент {len(features)} | оставлено {int(survivors.sum())} | "
        f"шаблонных {int(repeated.sum())} | размера текста {int((text_sized & ~enclosed).sum())}"
    )
    print(
        f"габарит текста {text_scale:.0f} px | больших контуров {int(hosts.sum())}, внутри них {int(enclosed.sum())} компонент | "
        f"ребер оставлено {int(kept_paths.sum())} из {graph.path_count}"
    )
    draw_component_selection(
        graph,
        features,
        kept_paths,
        scaled_image,
        build_view_window(scaled_image.shape[:2], center_x, center_y, zoom),
        f"правило: внутри >= {minimum_inside_area}, разброс t <= {maximum_thickness_spread:.1f}, извил >= {minimum_tortuosity:.2f}",
    )
    return kept_paths


ui_rules = interactive(
    run_component_rules,
    **window_sliders(),
    minimum_inside_area=IntSlider(
        value=1000, min=0, max=20000, step=500, continuous_update=False, description="внутри >="
    ),
    maximum_thickness_spread=FloatSlider(
        value=2.5, min=1.1, max=6.0, step=0.1, continuous_update=False, description="разброс t"
    ),
    minimum_tortuosity=FloatSlider(
        value=1.6, min=1.0, max=3.0, step=0.05, continuous_update=False, description="извил >="
    ),
    repeat_tolerance=FloatSlider(
        value=0.05, min=0.01, max=0.30, step=0.01, continuous_update=False, description="допуск"
    ),
    minimum_repeat_count=IntSlider(value=3, min=2, max=10, step=1, continuous_update=False, description="повторов"),
    text_size_quantile=FloatSlider(
        value=0.5, min=0.0, max=1.0, step=0.05, continuous_update=False, description="квантиль"
    ),
    text_size_tolerance=FloatSlider(
        value=1.0, min=0.0, max=3.0, step=0.1, continuous_update=False, description="размер текста"
    ),
    host_area_share=FloatSlider(
        value=0.2, min=0.05, max=1.0, step=0.05, continuous_update=False, description="тело детали"
    ),
)
ui_rules

interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='центр X', max=1.0, step=0.0…

## Обрезка хвостов

Наконечник приклеен к детали и живет внутри ее компоненты, поэтому правилом по компонентам он не
снимается. Порог относительный: хвост режется, если он короче заданной доли длины своей компоненты.

In [6]:
def run_trim(center_x: float, center_y: float, zoom: float, twig_length_share: float) -> Mask:
    """Слайдер обрезки хвостов."""
    alive_paths = ui_rules.result
    final_paths = trim_short_twigs(graph, alive_paths, features.path_component, twig_length_share)

    trimmed = alive_paths & ~final_paths
    print(f"срезано хвостов {int(trimmed.sum())} | длина {graph.length[trimmed].sum():.0f} px")
    print(f"ребер в итоге {int(final_paths.sum())} из {graph.path_count}")
    draw_trimmed_twigs(
        graph,
        ~alive_paths,
        trimmed,
        final_paths,
        scaled_image,
        build_view_window(scaled_image.shape[:2], center_x, center_y, zoom),
        f"хвосты короче {twig_length_share:.0%} длины своей компоненты",
    )
    return final_paths


ui_trim = interactive(
    run_trim,
    **window_sliders(),
    twig_length_share=FloatSlider(
        value=0.05, min=0.0, max=0.3, step=0.01, continuous_update=False, description="хвост <"
    ),
)
ui_trim

interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='центр X', max=1.0, step=0.0…

## Итоговая маска детали

Пиксели берутся из оригинальной бинаризации, а не из замкнутой маски: перемычки нужны были только для
анализа связности.

In [7]:
def run_part_mask(radius_tolerance: float) -> Mask:
    """Слайдер перерисовки итоговой маски."""
    mask = paint_selected_paths(graph, ui_trim.result, original_strokes_mask, radius_tolerance)
    metrics = measure_mask_metrics(mask)
    print(f"чернил {mask.sum() / original_strokes_mask.sum():.1%} от оригинала | кусков {metrics.component_count}")
    show_mask(mask, f"деталь без мусора, допуск {radius_tolerance:.1f} px")
    return mask


ui_part_mask = interactive(
    run_part_mask,
    radius_tolerance=FloatSlider(value=0.5, min=0.0, max=2.0, step=0.1, continuous_update=False, description="допуск"),
)
ui_part_mask

interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='допуск', max=2.0), Output()…

## Результат

In [8]:
part_mask: Mask = ui_part_mask.result
stage_parameters = {
    "source_stage": stored_closed.path.name,
    **{name: value for name, value in ui_rules.kwargs.items() if name not in ("center_x", "center_y", "zoom")},
    **{name: value for name, value in ui_trim.kwargs.items() if name not in ("center_x", "center_y", "zoom")},
    **ui_part_mask.kwargs,
}

saved_path = cache.save(source_path, "part_mask", stage_parameters, part_mask=part_mask)
print(f"сохранено: {saved_path}")
print(f"параметры: {stage_parameters}")

сохранено: .cache/9/part_mask__2c8c9de24f.npz
параметры: {'source_stage': 'closed_mask__6846bc5552.npz', 'minimum_inside_area': 1000, 'maximum_thickness_spread': 2.5, 'minimum_tortuosity': 1.6, 'repeat_tolerance': 0.05, 'minimum_repeat_count': 3, 'text_size_quantile': 0.5, 'text_size_tolerance': 1.0, 'twig_length_share': 0.05, 'radius_tolerance': 0.5}
